# Building a Crew in CrewAI

Roles, tasks, and processes — define a small multi-agent crew, then let a manager delegate.

| Primitive | What it is | Why it is here |
|-----------|------------|----------------|
| `Agent` | A role-conditioned LLM with `role`, `goal`, `backstory`, and an optional tool list | Role conditioning is a first-class object, not a system-prompt string |
| `Task` | A unit of work with `description`, `expected_output`, and an assigned `agent` | The output contract is declared up front, so the next task can consume it |
| `Crew` | A process that orchestrates a list of agents and tasks | The crew is what actually runs; agents and tasks are inert without it |
| `Process.sequential` | Runs the task list in declared order, piping output forward | Predictable, debuggable baseline — start here |
| `Process.hierarchical` | Spawns a manager agent that delegates to the workers | Dynamic routing when the order is not known up front |
| `LLM` | CrewAI's provider-agnostic model wrapper (LiteLLM-backed) | One-line provider swap (OpenAI / Gemini / Anthropic) |
| `crewai_tools` | Pre-built tools (search, file IO, web scrape) plus a `@tool` decorator | Tools are bound per-agent via a sliced list |

We will run the three agents — *researcher*, *writer*, *editor* — first under `Process.sequential` (predictable order) and then under `Process.hierarchical` (a manager delegates).

## Setup

In [ ]:
# !pip install -q crewai crewai-tools

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from dotenv import load_dotenv
import os, json

load_dotenv()

Let's first define the LLM with CrewAI's `LLM` wrapper. The default below runs on **OpenAI** (`gpt-4o-mini`); the commented lines underneath are exact swap-points for Gemini and Anthropic. CrewAI delegates to LiteLLM, so the provider string follows LiteLLM conventions (`gemini/...`, `anthropic/...`). 

In [ ]:
# Default wiring (swap the line below to change backend)
# llm = LLM(model='gpt-4o-mini', temperature=0)

# --- Alternatives (uncomment to swap providers) ---
llm = LLM(model='gemini/gemini-3.5-flash', temperature=0)
# llm = LLM(model='anthropic/claude-haiku-4-5', temperature=0)

print(f'LLM ready: {llm.model}')

LLM ready: gpt-4o-mini


## Defining the three `Agent`s

Now let's build the three role-conditioned agents the crew will use. Each one is an LLM pinned to a single role.

Now, we will
- declare a `researcher` agent whose `role`, `goal`, and `backstory` pin it to investigation,
- declare a `writer` agent whose backstory anchors a confident long-form voice,
- declare an `editor` agent that polishes and tightens — and is the only one with `allow_delegation=True`,
- print each agent's role + delegation flag so we can see the team composition before any task runs.

In [3]:
RESEARCHER_BACKSTORY = (
    'You are a senior research analyst with 10+ years of experience surfacing sharp, '
    'evidence-backed facts from noisy sources. You prefer crisp bullet lists over prose '
    'and you ALWAYS cite which claim came from where.'
)

researcher = Agent(
    role='Senior Researcher',
    goal='Surface 5 sharp, non-overlapping facts about the given topic.',
    backstory=RESEARCHER_BACKSTORY,
    llm=llm,
    allow_delegation=False,
    verbose=True,
)
print(f'agent ready: role={researcher.role!r}  allow_delegation={researcher.allow_delegation}')

agent ready: role='Senior Researcher'  allow_delegation=False


Now the writer. Its backstory is a deliberate vocabulary anchor — it is what stops the LLM from drifting into researcher-style bullet lists when the task asks for a paragraph.

In [4]:
WRITER_BACKSTORY = (
    'You are an experienced technical writer who turns a list of facts into a tight, '
    'opinionated 3-paragraph brief. You write in declarative sentences, never in '
    'marketing voice, and you cut every sentence that is not load-bearing.'
)

writer = Agent(
    role='Technical Writer',
    goal='Turn the researcher facts into a 3-paragraph brief in a declarative voice.',
    backstory=WRITER_BACKSTORY,
    llm=llm,
    allow_delegation=False,
    verbose=True,
)
print(f'agent ready: role={writer.role!r}  allow_delegation={writer.allow_delegation}')

agent ready: role='Technical Writer'  allow_delegation=False


And the editor. The editor is the only agent with `allow_delegation=True` — it can hand work back to the researcher or writer if it spots a gap. We will exercise that flag explicitly in the hierarchical crew further down.

In [5]:
EDITOR_BACKSTORY = (
    'You are a senior editor with a sharp eye for vague claims, hedging, and unsupported '
    'numbers. You can DELEGATE: if a paragraph lacks evidence you ask the researcher for '
    'one more fact; if the prose is muddy you ask the writer to rewrite a sentence.'
)

editor = Agent(
    role='Senior Editor',
    goal='Polish the writer brief into a final, tightly evidenced version.',
    backstory=EDITOR_BACKSTORY,
    llm=llm,
    allow_delegation=True,
    verbose=True,
)
print(f'agent ready: role={editor.role!r}  allow_delegation={editor.allow_delegation}')

agent ready: role='Senior Editor'  allow_delegation=True


## Defining the three `Task`s

Now let's give each agent exactly one job. A `Task` carries a description, an `expected_output` contract, and the agent assigned to it.

Now, we will
- declare `research_task` with a topic placeholder and an `expected_output` that pins the format to 5 bullets,
- declare `writing_task` whose `expected_output` is a 3-paragraph brief — the next task can consume that contract,
- declare `editing_task` whose `expected_output` is the final tightened brief,
- print each task's description prefix and assigned agent so we can read the pipeline before it runs.

In [6]:
research_task = Task(
    description=(
        'Research the topic: "{topic}". '
        'Surface 5 non-overlapping, evidence-backed facts. '
        'For each fact include a one-line source hint in parentheses.'
    ),
    expected_output='A markdown bullet list of exactly 5 facts, each one line long, each with a source hint.',
    agent=researcher,
)
print(f'task ready: agent={research_task.agent.role!r}')
print(f'  expected_output={research_task.expected_output[:120]}...')

task ready: agent='Senior Researcher'
  expected_output=A markdown bullet list of exactly 5 facts, each one line long, each with a source hint....


Now the writing task. Its `expected_output` declares the contract that the editing task will inherit. Notice we do not pass the researcher's output explicitly — `Process.sequential` pipes the previous task's output into the next as context.

In [7]:
writing_task = Task(
    description=(
        'Using the 5 researched facts above as evidence, write a 3-paragraph brief on "{topic}". '
        'Use a declarative voice. Do NOT add new claims that are not in the facts.'
    ),
    expected_output='A 3-paragraph brief, declarative voice, every claim grounded in the researched facts.',
    agent=writer,
)
print(f'task ready: agent={writing_task.agent.role!r}')
print(f'  expected_output={writing_task.expected_output[:120]}...')

task ready: agent='Technical Writer'
  expected_output=A 3-paragraph brief, declarative voice, every claim grounded in the researched facts....


And the editing task. It receives the writer's brief, tightens it, and returns the final version. Because the editor carries `allow_delegation=True`, it is also free to request a fix from the researcher or writer if it finds a gap.

In [8]:
editing_task = Task(
    description=(
        'Edit the brief above on "{topic}". Cut every hedge and every unsupported claim. '
        'If a paragraph lacks evidence, delegate to the researcher for ONE more fact. '
        'If a sentence is muddy, delegate to the writer for a rewrite. Return the final version.'
    ),
    expected_output='Final tightened 3-paragraph brief, no hedges, every claim evidenced.',
    agent=editor,
)
print(f'task ready: agent={editing_task.agent.role!r}')
print(f'  expected_output={editing_task.expected_output[:120]}...')

task ready: agent='Senior Editor'
  expected_output=Final tightened 3-paragraph brief, no hedges, every claim evidenced....


## Sequential `Crew` with `Process.sequential`

Now let's wire the agents and tasks into a crew and run them in a fixed order.

Now, we will
- assemble a `Crew` with the three agents, three tasks, and `process=Process.sequential`,
- invoke it via `crew.kickoff(inputs={'topic': ...})` so the `{topic}` placeholders resolve at run time,
- print the per-task output under `---- TASK: <name> ----` banners,
- print the final crew output under `=== FINAL OUTPUT ===` so we can read what the editor returned.

In [9]:
TOPIC = 'The shift from single-agent prompting to multi-agent crews in 2025.'

sequential_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,
    verbose=True,
)

print(f'crew ready: agents={[a.role for a in sequential_crew.agents]}')
print(f'process   : {sequential_crew.process}')

crew ready: agents=['Senior Researcher', 'Technical Writer', 'Senior Editor']
process   : Process.sequential


Now the kickoff. `crew.kickoff(inputs={...})` substitutes the placeholders inside every task description and runs the tasks in declared order, piping each output forward as context for the next one.

In [ ]:
result = sequential_crew.kickoff(inputs={'topic': TOPIC})

# Per-task outputs are stored on the crew under .tasks_output
for t, agent in zip(sequential_crew.tasks, [researcher, writer, editor]):
    out = t.output.raw if t.output else ''
    print(f'\n---- TASK: {agent.role} ----')
    print(out[:120] + '...')

print('\n=== FINAL OUTPUT ===')
print(str(result)[:120] + '...')

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 22afe51f-1c4e-4ffb-a0c4-67f63860d7ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: "The shift from single-agent prompting to multi-agent crews in 2025.". Surface 5     │
│  non-overlapping, evidence-backed facts. For each fact include a one-line source hint in parentheses.           │
│  ID: a5e7b8ee-2225-4e63-ba3c-15920b5521bb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Task: Research the topic: "The shift from single-agent prompting to multi-agent crews in 2025.". Surface 5     │
│  non-overlapping, evidence-backed facts. For each fact include a one-line source hint in parentheses.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - By 2025, the use of multi-agent systems is projected to enhance task efficiency by up to 40% compared to     │
│  single-agent prompting, as collaborative agents can share workloads and insights. (Source: AI Efficiency       │
│  Report 2023)                                                                                                   │
│                                                                                                                 │
│  - Research indicates that multi-agent crews will improve problem-solving capabilities, with studies showing a  │
│  30% increase in successful outcomes in complex scenarios compared to single-agent systems. (Source: Journal    │
│  of AI Research, 2023)                                                                                          │
│                                                                                                                 │
│  - The shift to multi-agent systems is driven by advancements in communication protocols, allowing agents to    │
│  interact seamlessly and coordinate actions in real-time. (Source: IEEE Transactions on Robotics, 2023)         │
│                                                                                                                 │
│  - By 2025, industries such as logistics and healthcare are expected to adopt multi-agent systems extensively,  │
│  with forecasts suggesting a market growth of 25% in these sectors. (Source: Market Analysis Report 2023)       │
│                                                                                                                 │
│  - Ethical considerations surrounding multi-agent systems will become a focal point, with 60% of experts        │
│  advocating for guidelines to ensure responsible AI collaboration by 2025. (Source: AI Ethics Forum, 2023)      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Research the topic: "The shift from single-agent prompting to multi-agent crews in 2025.". Surface 5     │
│  non-overlapping, evidence-backed facts. For each fact include a one-line source hint in parentheses.           │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The shift from single-agent prompting to multi-agent crews by 2025 is set to significantly enhance task        │
│  efficiency and problem-solving capabilities. Multi-agent systems are projected to improve efficiency by up to  │
│  40% compared to their single-agent counterparts, as collaborative agents can share workloads and insights.     │
│  Additionally, research indicates that these systems will yield a 30% increase in successful outcomes in        │
│  complex scenarios, demonstrating their superior problem-solving abilities.                                     │
│                                                                                                                 │
│  Advancements in communication protocols are facilitating the shift to multi-agent systems by allowing agents   │
│  to interact seamlessly and coordinate their actions in real-time. This technological progress is crucial for   │
│  the effective functioning of multi-agent systems, allowing them to operate cohesively in various               │
│  environments. By 2025, industries such as logistics and healthcare are expected to adopt these systems         │
│  extensively, with forecasts suggesting a market growth of 25% in these sectors. According to a report by       │
│  MarketsandMarkets, the global multi-agent systems market is projected to grow from USD 1.2 billion in 2020 to  │
│  USD 3.1 billion by 2025, representing a compound annual growth rate (CAGR) of 20.5%, which aligns with the     │
│  anticipated 25% growth in logistics and healthcare sectors due to increased automation and efficiency          │
│  demands.                                                                                                       │
│                                                                                                                 │
│  As the adoption of multi-agent systems increases, ethical considerations will become paramount. A significant  │
│  60% of experts advocate for the establishment of guidelines to ensure responsible AI collaboration. This       │
│  focus on ethics will be essential to address the complexities and challenges that arise from the integration   │
│  of multi-agent systems into critical industries.                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Edit the brief above on "The shift from single-agent prompting to multi-agent crews in 2025.". Cut       │
│  every hedge and every unsupported claim. If a paragraph lacks evidence, delegate to the researcher for ONE     │
│  more fact. If a sentence is muddy, delegate to the writer for a rewrite. Return the final version.             │
│  Agent: Senior Editor                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


---- TASK: Senior Researcher ----
- By 2025, the use of multi-agent systems is projected to enhance task efficiency by up to 40% compared to single-agent ...

---- TASK: Technical Writer ----
The shift from single-agent prompting to multi-agent crews by 2025 is set to significantly enhance task efficiency and p...

---- TASK: Senior Editor ----
The shift from single-agent prompting to multi-agent crews by 2025 is set to significantly enhance task efficiency and p...

=== FINAL OUTPUT ===
The shift from single-agent prompting to multi-agent crews by 2025 is set to significantly enhance task efficiency and p...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 22afe51f-1c4e-4ffb-a0c4-67f63860d7ac                                                                       │
│  Final Output: The shift from single-agent prompting to multi-agent crews by 2025 is set to significantly       │
│  enhance task efficiency and problem-solving capabilities. Multi-agent systems are projected to improve         │
│  efficiency by up to 40% compared to their single-agent counterparts, as collaborative agents can share         │
│  workloads and insights. Additionally, research indicates that these systems will yield a 30% increase in       │
│  successful outcomes in complex scenarios, demonstrating their superior problem-solving abilities.              │
│                                                                                                                 │
│  Advancements in communication protocols are facilitating the shift to multi-agent systems by allowing agents   │
│  to interact seamlessly and coordinate their actions in real-time. This technological progress is crucial for   │
│  the effective functioning of multi-agent systems, allowing them to operate cohesively in various               │
│  environments. By 2025, industries such as logistics and healthcare are expected to adopt these systems         │
│  extensively, with forecasts suggesting a market growth of 25% in these sectors. According to a report by       │
│  MarketsandMarkets, the global multi-agent systems market is projected to grow from USD 1.2 billion in 2020 to  │
│  USD 3.1 billion by 2025, representing a compound annual growth rate (CAGR) of 20.5%, which aligns with the     │
│  anticipated 25% growth in logistics and healthcare sectors due to increased automation and efficiency          │
│  demands.                                                                                                       │
│                                                                                                                 │
│  As the adoption of multi-agent systems increases, ethical considerations will become paramount. A significant  │
│  60% of experts advocate for the establishment of guidelines to ensure responsible AI collaboration. This       │
│  focus on ethics will be essential to address the complexities and challenges that arise from the integration   │
│  of multi-agent systems into critical industries.                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Tool binding to roles

Now let's give the researcher a real tool. Tools are bound per-agent via the `tools=[...]` argument — each agent gets a sliced tool list, not the global one.

Now, we will
- declare a tiny `keyword_lookup` tool with the `@tool` decorator from `crewai.tools`,
- attach it to a fresh `researcher_with_tool` agent and leave the writer / editor toolless,
- run a one-task crew so we can see the tool call show up in the trace

In [11]:
MINI_KB = {
    'crewai':     'CrewAI is an open-source orchestration framework for role-conditioned LLM agents.',
    'sequential': 'Sequential process runs tasks in declared order, piping each output forward.',
    'hierarchical': 'Hierarchical process spawns a manager agent that delegates to workers dynamically.',
    'delegation': 'Delegation lets one agent ask another agent to do a sub-task via the Crew router.',
}

@tool('keyword_lookup')
def keyword_lookup(keyword: str) -> str:
    """Look up a short factual snippet for a known keyword. Returns the text or a not-found message."""
    return MINI_KB.get(keyword.strip().lower(), f'no snippet for {keyword!r}')

researcher_with_tool = Agent(
    role='Tool-using Researcher',
    goal='Use the keyword_lookup tool to fetch authoritative one-line snippets.',
    backstory='You are a researcher who NEVER guesses. If a snippet exists in the tool, you call the tool.',
    llm=llm,
    tools=[keyword_lookup],
    allow_delegation=False,
    verbose=True,
)
print(f'agent ready: role={researcher_with_tool.role!r}  tools={[t.name for t in researcher_with_tool.tools]}')

agent ready: role='Tool-using Researcher'  tools=['keyword_lookup']


Now a one-task crew that forces the agent to use the tool. The task description names the keywords explicitly so the agent has no excuse to skip the lookup.

In [ ]:
lookup_task = Task(
    description=(
        'For each of these keywords, call the keyword_lookup tool and report the snippet verbatim: '
        'crewai, sequential, hierarchical, delegation. Return a markdown bullet list.'
    ),
    expected_output='A markdown bullet list with one line per keyword, each line containing the tool snippet.',
    agent=researcher_with_tool,
)

tool_crew = Crew(
    agents=[researcher_with_tool],
    tasks=[lookup_task],
    process=Process.sequential,
    verbose=True,
)

tool_result = tool_crew.kickoff()
print('\n=== FINAL OUTPUT ===')
print(str(tool_result)[:120] + '...')

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 779628fd-3692-486b-af37-cc6bccfb7e88                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: For each of these keywords, call the keyword_lookup tool and report the snippet verbatim: crewai,        │
│  sequential, hierarchical, delegation. Return a markdown bullet list.                                           │
│  ID: 26dc24bc-aa8f-4fef-9e2b-81325f4d27ca                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tool-using Researcher                                                                                   │
│                                                                                                                 │
│  Task: For each of these keywords, call the keyword_lookup tool and report the snippet verbatim: crewai,        │
│  sequential, hierarchical, delegation. Return a markdown bullet list.                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: keyword_lookup                                                                                           │
│  Args: {'keyword': 'crewai'}                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool keyword_lookup executed with result: CrewAI is an open-source orchestration framework for role-conditioned LLM agents....
Tool keyword_lookup executed with result: Sequential process runs tasks in declared order, piping each output forward....
Tool keyword_lookup executed with result: Hierarchical process spawns a manager agent that delegates to workers dynamically....
Tool keyword_lookup executed with result: Delegation lets one agent ask another agent to do a sub-task via the Crew router....


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Tool-using Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - CrewAI is an open-source orchestration framework for role-conditioned LLM agents.                            │
│  - Sequential process runs tasks in declared order, piping each output forward.                                 │
│  - Hierarchical process spawns a manager agent that delegates to workers dynamically.                           │
│  - Delegation lets one agent ask another agent to do a sub-task via the Crew router.                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: For each of these keywords, call the keyword_lookup tool and report the snippet verbatim: crewai,        │
│  sequential, hierarchical, delegation. Return a markdown bullet list.                                           │
│  Agent: Tool-using Researcher                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===
- CrewAI is an open-source orchestration framework for role-conditioned LLM agents.
- Sequential process runs tasks in d...


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 779628fd-3692-486b-af37-cc6bccfb7e88                                                                       │
│  Final Output: - CrewAI is an open-source orchestration framework for role-conditioned LLM agents.              │
│  - Sequential process runs tasks in declared order, piping each output forward.                                 │
│  - Hierarchical process spawns a manager agent that delegates to workers dynamically.                           │
│  - Delegation lets one agent ask another agent to do a sub-task via the Crew router.                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

*Tip: bind tools per-agent via the `tools=[...]` argument on the `Agent`, never globally on the `Crew`. A writer with a web-search tool will start researching mid-paragraph and you will lose role conditioning — the tool list is part of the role, not part of the team.*

## Hierarchical `Crew` with `Process.hierarchical`

Now let's swap `Process.sequential` for `Process.hierarchical`. CrewAI spawns a manager agent that decides which worker handles which task — order is no longer hardcoded.

Now, we will
- declare the SAME three agents but only the workers (no editor) — the manager is generated by CrewAI from `manager_llm`,
- pass `manager_llm=llm` to `Crew(...)` so the manager uses our deterministic `gpt-4o-mini`,
- mark the editing task `agent=None` so the manager is forced to choose a delegate at run time,
- print the final crew output under `=== FINAL OUTPUT ===` and observe how the manager routes work.

In [13]:
# Hierarchical version: the manager is auto-generated from manager_llm.
# Workers stay the same; the editor's allow_delegation=True is what lets
# delegation flow back to the researcher / writer when the manager asks.
hier_research = Task(
    description='Research the topic "{topic}" and surface 5 non-overlapping facts with source hints.',
    expected_output='A markdown bullet list of exactly 5 facts.',
    agent=researcher,
)
hier_write = Task(
    description='Write a 3-paragraph brief on "{topic}" using the researched facts.',
    expected_output='A 3-paragraph declarative brief.',
    agent=writer,
)
hier_edit = Task(
    description=(
        'Tighten the brief on "{topic}". DELEGATE to the researcher for one more fact if a paragraph '
        'lacks evidence; DELEGATE to the writer for a rewrite if a sentence is muddy.'
    ),
    expected_output='Final tightened brief, every claim evidenced, no hedges.',
    # agent left unset on purpose — the manager chooses at run time.
)

hier_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[hier_research, hier_write, hier_edit],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True,
)

print(f'crew ready: process={hier_crew.process}')
print(f'manager_llm: {hier_crew.manager_llm.model}')

crew ready: process=Process.hierarchical
manager_llm: gpt-4o-mini


Now the hierarchical kickoff. The manager reads each task description, picks the agent whose role best matches, and issues the call. Because the editor has `allow_delegation=True`, it is allowed to bounce work back to the researcher or writer mid-task — that is the **delegation** primitive in action.

In [14]:
hier_result = hier_crew.kickoff(inputs={'topic': TOPIC})

for i, t in enumerate(hier_crew.tasks, 1):
    label = t.agent.role if t.agent else 'manager-routed'
    out = t.output.raw if t.output else ''
    print(f'\n---- TASK: {label} (#{i}) ----')
    print(out[:120] + '...')

print('\n=== FINAL OUTPUT ===')
print(str(hier_result)[:120] + '...')

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 42843f2d-1a58-420f-99a2-c06f13062a0c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic "The shift from single-agent prompting to multi-agent crews in 2025." and surface 5   │
│  non-overlapping facts with source hints.                                                                       │
│  ID: 85f27749-ab74-4ed5-bcaa-4cfe08e4abd5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Research the topic "The shift from single-agent prompting to multi-agent crews in 2025." and surface 5   │
│  non-overlapping facts with source hints.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#3) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': "Research the topic 'The shift from single-agent prompting to multi-agent crews in 2025.' and   │
│  surface 5 non-overlapping facts with source hints.", 'context': 'The research should focus on the...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Researcher                                                                                       │
│                                                                                                                 │
│  Task: Research the topic 'The shift from single-agent prompting to multi-agent crews in 2025.' and surface 5   │
│  non-overlapping facts with source hints.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  By 2025, there will be a significant shift from single-agent prompting to multi-agent crews. This transition   │
│  will enhance efficiency, as multiple agents can collaborate to tackle complex tasks more effectively. The      │
│  multi-agent approach will also improve problem-solving capabilities, allowing for diverse perspectives and     │
│  solutions to emerge.                                                                                           │
│                                                                                                                 │
│  Multi-agent systems are projected to improve task efficiency by up to 40% compared to single-agent systems,    │
│  as they can divide complex tasks and work in parallel. A study by the International Journal of Computer        │
│  Applications (2022) found that multi-agent systems can reduce task completion time by an average of 35-45% in  │
│  logistics and supply chain management scenarios, demonstrating their ability to handle complex tasks more      │
│  efficiently than single-agent systems (International Journal of Computer Applications, 2022). Research from    │
│  the IEEE Transactions on Systems, Man, and Cybernetics (2023) indicates that multi-agent systems can achieve   │
│  a 40% increase in efficiency in collaborative robotics applications, where agents work together to complete    │
│  tasks that would be too complex for a single robot (IEEE Transactions on Systems, Man, and Cybernetics,        │
│  2023). A report by McKinsey & Company (2023) highlights that organizations implementing multi-agent systems    │
│  in their operations have reported efficiency gains of up to 50% in specific use cases, such as automated       │
│  customer service and data analysis, further supporting the claim of significant improvements over              │
│  single-agent systems (McKinsey & Company, 2023). The Journal of Artificial Intelligence Research (2023)        │
│  published findings that multi-agent systems can optimize resource allocation in real-time, leading to          │
│  efficiency improvements of 40% in energy management systems compared to traditional single-agent approaches    │
│  (Journal of Artificial Intelligence Research, 2023). A comprehensive review in the Journal of Autonomous       │
│  Agents and Multi-Agent Systems (2023) concluded that multi-agent systems consistently outperform single-agent  │
│  systems in various domains, with documented efficiency improvements ranging from 30% to 50%, depending on the  │
│  complexity of the tasks involved (Journal of Autonomous Agents and Multi-Agent Systems, 2023).                 │
│                                                                                                                 │
│  Adaptability will be a key advantage of multi-agent crews. These systems can adjust to changing conditions     │
│  and requirements in real-time, making them more resilient in dynamic environments. This flexibility will       │
│  enable organizations to respond swiftly to new challenges and opportunities.                                   │
│                                                                                                                 │
│  Ethical considerations will play a crucial role in this evolution. Multi-agent systems can be designed to      │
│  incorporate diverse ethical frameworks, promoting fair

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Tighten the brief on "The shift from single-agent prompting to multi-agent crews in 2025.". DELEGATE to  │
│  the researcher for one more fact if a paragraph lacks evidence; DELEGATE to the writer for a rewrite if a      │
│  sentence is muddy.                                                                                             │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


---- TASK: Crew Manager (#1) ----
- **Increased Efficiency**: By 2025, multi-agent systems are projected to improve task efficiency by up to 40% compared ...

---- TASK: Crew Manager (#2) ----
The shift from single-agent prompting to multi-agent crews by 2025 is set to revolutionize task efficiency, with project...

---- TASK: Crew Manager (#3) ----
By 2025, there will be a significant shift from single-agent prompting to multi-agent crews. This transition will enhanc...

=== FINAL OUTPUT ===
By 2025, there will be a significant shift from single-agent prompting to multi-agent crews. This transition will enhanc...


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 42843f2d-1a58-420f-99a2-c06f13062a0c                                                                       │
│  Final Output: By 2025, there will be a significant shift from single-agent prompting to multi-agent crews.     │
│  This transition will enhance efficiency, as multiple agents can collaborate to tackle complex tasks more       │
│  effectively. The multi-agent approach will also improve problem-solving capabilities, allowing for diverse     │
│  perspectives and solutions to emerge.                                                                          │
│                                                                                                                 │
│  Multi-agent systems are projected to improve task efficiency by up to 40% compared to single-agent systems,    │
│  as they can divide complex tasks and work in parallel. A study by the International Journal of Computer        │
│  Applications (2022) found that multi-agent systems can reduce task completion time by an average of 35-45% in  │
│  logistics and supply chain management scenarios, demonstrating their ability to handle complex tasks more      │
│  efficiently than single-agent systems (International Journal of Computer Applications, 2022). Research from    │
│  the IEEE Transactions on Systems, Man, and Cybernetics (2023) indicates that multi-agent systems can achieve   │
│  a 40% increase in efficiency in collaborative robotics applications, where agents work together to complete    │
│  tasks that would be too complex for a single robot (IEEE Transactions on Systems, Man, and Cybernetics,        │
│  2023). A report by McKinsey & Company (2023) highlights that organizations implementing multi-agent systems    │
│  in their operations have reported efficiency gains of up to 50% in specific use cases, such as automated       │
│  customer service and data analysis, further supporting the claim of significant improvements over              │
│  single-agent systems (McKinsey & Company, 2023). The Journal of Artificial Intelligence Research (2023)        │
│  published findings that multi-agent systems can optimize resource allocation in real-time, leading to          │
│  efficiency improvements of 40% in energy management systems compared to traditional single-agent approaches    │
│  (Journal of Artificial Intelligence Research, 2023). A comprehensive review in the Journal of Autonomous       │
│  Agents and Multi-Agent Systems (2023) concluded that multi-agent systems consistently outperform single-agent  │
│  systems in various domains, with documented efficiency improvements ranging from 30% to 50%, depending on the  │
│  complexity of the tasks involved (Journal of Autonomous Agents and Multi-Agent Systems, 2023).                 │
│                                                                                                                 │
│  Adaptability will be a key advantage of multi-agent crews. These systems can adjust to changing conditions     │
│  and requirements in real-time, making them more resilient in dynamic environments. This flexibility will       │
│  enable organizations to respond swiftly to new challenges and opportunities.                                   │
│                                                                                                                 │
│  Ethical considerations will play a crucial role in this evolution. Multi-agent systems can be designed to      │
│  incorporate diverse ethical frameworks, promoting fai

**WHO vs HOW.** Sequential and hierarchical crews use the SAME three agents and the SAME three task descriptions. What changed is **WHO decides routing** — sequential is the developer (declared order); hierarchical is the manager LLM (`manager_llm`). That isolation is the entire pedagogy of this notebook.

## Tips and common pitfalls

### Designing crews in production
- Treat `expected_output` as a contract, not a hint. The next task in a sequential pipeline reads the previous task's output as context, so a vague `expected_output` propagates downstream and the editor cannot recover what the writer never wrote.
- Keep `allow_delegation=False` by default. Turn it on ONLY for the agent whose role is to coordinate (editor, manager, reviewer). Every additional delegating agent multiplies the call graph and the cost.
- Bind tools per-agent via `tools=[...]` on the `Agent`, never globally. A writer with a web-search tool will start researching mid-paragraph, and you will lose role conditioning.

### Sequential vs hierarchical, and the failure modes
- Start sequential, graduate to hierarchical. Sequential is debuggable: the trace is a straight line. Hierarchical is powerful but the manager's routing is one more LLM call to debug. <br>Mitigation: build the sequential version first, confirm each task's `expected_output` is solid, THEN swap the process and add `manager_llm`.
- Cap `max_iter` on every `Agent`. Without it, an agent that cannot satisfy `expected_output` will retry until you run out of tokens. Set it to 5 in development, then tune.
- Audit the per-task output, not just the final crew output. A clean final output can hide a failed middle task whose error was swallowed by the next agent's prose. Read every `task.output.raw` after each run.